# Day 7: Training Loop — Learning from Data

**Learning Objective**: Implement a complete training loop with loss computation and gradient descent.

Today, we'll make our network **learn** by implementing:
- Loss function (Mean Squared Error)
- Gradient descent (update parameters)
- Training loop (iterate until convergence)

This is the culmination of Week 1! 🎉

In [ ]:
import math
import random

## Theory

### Loss Function (Mean Squared Error)
$$L = \frac{1}{n}\sum_{i=1}^{n}(y_{pred}^{(i)} - y_{true}^{(i)})^2$$

### Gradient Descent
$$w_{new} = w_{old} - \eta \cdot \frac{\partial L}{\partial w}$$

Where $\eta$ is the **learning rate** (typically 0.01 to 0.1).

### Why Zero Gradients?
Gradients accumulate with `+=`. If we don't zero them before each iteration, gradients from previous iterations add up!

## Neural Network Code (from Days 4-6)

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return Value(other) * self**-1

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
    
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Training Data

Simple classification: is the sum of inputs positive?

In [ ]:
# Training data: is sum of inputs positive?
xs = [
    [Value(2.0), Value(3.0)],    # sum = 5  → target: 1
    [Value(3.0), Value(-1.0)],   # sum = 2  → target: 1
    [Value(-1.0), Value(-2.0)],  # sum = -3 → target: -1
    [Value(1.0), Value(1.0)],    # sum = 2  → target: 1
]
ys = [Value(1.0), Value(1.0), Value(-1.0), Value(1.0)]

print("Training data:")
for x, y in zip(xs, ys):
    print(f"  Input: [{x[0].data}, {x[1].data}] → Target: {y.data}")

## Loss Function (Mean Squared Error)

In [ ]:
def compute_loss(model, xs, ys):
    """Compute Mean Squared Error loss."""
    # Forward pass for all examples
    predictions = [model(x) for x in xs]
    
    # Compute squared errors
    losses = [(pred - yt)**2 for pred, yt in zip(predictions, ys)]
    
    # Mean loss
    total_loss = sum(losses) * (1.0 / len(losses))
    return total_loss

# Test
mlp = MLP(2, [4, 1])
loss = compute_loss(mlp, xs, ys)
print(f"Initial loss: {loss.data:.4f}")

## Training Step

In [ ]:
def train_step(model, xs, ys, lr=0.05):
    """One step of gradient descent."""
    # 1. Forward pass & compute loss
    loss = compute_loss(model, xs, ys)
    
    # 2. Zero all gradients
    for p in model.parameters():
        p.grad = 0.0
    
    # 3. Backward pass
    loss.backward()
    
    # 4. Update parameters
    for p in model.parameters():
        p.data -= lr * p.grad
    
    return loss.data

# Test one step
mlp = MLP(2, [4, 1])
loss_before = compute_loss(mlp, xs, ys).data
loss_after = train_step(mlp, xs, ys)
print(f"Loss: {loss_before:.4f} → {loss_after:.4f}")
assert loss_after < loss_before, "Loss should decrease!"
print("✅ Training step works!")

## Full Training Loop

In [ ]:
# Create fresh network
mlp = MLP(2, [4, 4, 1])

# Training hyperparameters
epochs = 100
learning_rate = 0.05

# Training loop
print("Training...")
print("=" * 40)
for epoch in range(epochs):
    loss = train_step(mlp, xs, ys, lr=learning_rate)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: loss = {loss:.6f}")

print("=" * 40)
print(f"Final loss: {loss:.6f}")

## Check Predictions

In [ ]:
print("\n" + "=" * 50)
print("Final Predictions")
print("=" * 50)

correct = 0
for x, y in zip(xs, ys):
    pred = mlp(x)
    input_vals = [xi.data for xi in x]
    is_correct = (pred.data > 0) == (y.data > 0)
    correct += is_correct
    status = "✅" if is_correct else "❌"
    print(f"Input: {input_vals}, Target: {y.data:+.1f}, Pred: {pred.data:+.4f} {status}")

print(f"\nAccuracy: {correct}/{len(xs)} = {100*correct/len(xs):.0f}%")

## Challenge: Learn XOR

XOR is a classic problem that requires a hidden layer to solve.

In [ ]:
# XOR dataset
xs_xor = [
    [Value(0.0), Value(0.0)],  # 0 XOR 0 = 0 → -1
    [Value(0.0), Value(1.0)],  # 0 XOR 1 = 1 → +1
    [Value(1.0), Value(0.0)],  # 1 XOR 0 = 1 → +1
    [Value(1.0), Value(1.0)],  # 1 XOR 1 = 0 → -1
]
ys_xor = [Value(-1.0), Value(1.0), Value(1.0), Value(-1.0)]

# Train
mlp_xor = MLP(2, [4, 4, 1])

print("Training XOR...")
for epoch in range(200):
    loss = train_step(mlp_xor, xs_xor, ys_xor, lr=0.1)
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: loss = {loss:.6f}")

# Results
print("\nXOR Results:")
for x, y in zip(xs_xor, ys_xor):
    pred = mlp_xor(x)
    correct = "✅" if (pred.data > 0) == (y.data > 0) else "❌"
    print(f"{[int(xi.data) for xi in x]} → {pred.data:+.3f} (target: {y.data:+.1f}) {correct}")

## Experiment: Learning Rate Effects

In [ ]:
def experiment_lr(lr, epochs=50):
    """Train with different learning rates."""
    mlp = MLP(2, [4, 1])
    losses = []
    for epoch in range(epochs):
        loss = train_step(mlp, xs, ys, lr=lr)
        losses.append(loss)
    return losses

# Try different learning rates
print("Learning Rate Experiment:")
print("=" * 40)
for lr in [0.001, 0.01, 0.1, 0.5]:
    losses = experiment_lr(lr)
    status = "✅" if losses[-1] < 0.1 else "⚠️" if losses[-1] < 1.0 else "❌"
    print(f"lr={lr}: final_loss={losses[-1]:.6f} {status}")

## Summary — Week 1 Complete! 🎉

You've built a complete neural network from scratch:

| Day | What We Built |
|-----|---------------|
| 1 | `Value` class with basic operations |
| 2 | Manual backpropagation |
| 3 | Automatic `backward()` with chain rule |
| 4 | Power, division, negation, subtraction |
| 5 | `Neuron` class with tanh activation |
| 6 | `Layer` and `MLP` classes |
| 7 | Training loop with gradient descent |

**You now understand what PyTorch does under the hood!**

---

*Previous: [Day 6 — Layers](./day_06_layers.ipynb)*  
*Next: Week 2 — Loss Functions & Optimization*